**`monitor_build`**

Live progress and timing analysis for a running build.

- Works during or after a run.
- Reads three sources, each optional:
  Snakemake logs, openplaces stage timers, and outputs on disk.
- Re-run any cell to refresh.


# Configure


In [ ]:
import glob
import json
import re
from datetime import datetime
from pathlib import Path

import pandas as pd

import openplaces as op
from openplaces.config import cfg
from openplaces.core.schema import AdminId
from openplaces.recipe import get_output_path, get_recipe_by_id

# The build to watch: a region id from the shared registry.
REGION = 'cheer-eastern-nc'
RECIPE = 'US_footprint-cheer-2026'
SNAKEMAKE_LOG_DIR = Path('../../.snakemake/log')
TIMER_LOG_ROOT = Path(cfg.get_dir('cache')) / '_logs'

# Where is the process?

Progress from outputs on disk.

- Works with any runner: Snakemake, a driver, or stages by hand.
- One row per pipeline milestone, with the first missing units.


In [ ]:
admin_ids = op.get_region_admin_ids(REGION)
stages = {
    'parcels ingested': 'US-NC_parcel-nconemap-2025'
    if REGION.endswith('nc')
    else 'US-TX_parcel-txgio-2025',
    'footprint geospine': 'US_footprint-geospine-2026',
    'parcel curated': 'US_parcel-openplaces-2026',
    'footprint curated': RECIPE,
}
rows = []
for label, rid in stages.items():
    r = get_recipe_by_id(rid)
    have = [c for c in admin_ids if get_output_path(r, AdminId(*c.split('-'))).exists()]
    rows.append(
        {
            'stage': label,
            'done': len(have),
            'of': len(admin_ids),
            'missing': ','.join(sorted(set(admin_ids) - set(have))[:6]),
        }
    )
pd.DataFrame(rows).set_index('stage')

# Stage timings (openplaces timers)

Every stage persists per-step timings when it finishes.

- Files: `_logs/US/{ST}/{CO}/_all/{admin}_{Stage}_{ts}.json`.
- Use them to find slow steps and slow counties.


In [ ]:
records = []
for f in glob.glob(str(TIMER_LOG_ROOT / '**' / '*.json'), recursive=True):
    try:
        d = json.load(open(f, encoding='utf-8'))
    except Exception:
        continue
    for s in d.get('steps', []):
        records.append(
            {
                'stage': d.get('name'),
                'admin_id': d.get('admin_id'),
                'step': s.get('label'),
                'seconds': s.get('duration_seconds', s.get('duration', 0.0)),
            }
        )
timings = pd.DataFrame(records)
print(
    f'{len(timings):,} step records'
    if len(timings)
    else 'no timer logs yet (they appear as stages finish)'
)

In [ ]:
if len(timings):
    display(
        timings.groupby('step')['seconds']
        .agg(['count', 'median', 'max'])
        .sort_values('max', ascending=False)
        .head(15)
    )

In [ ]:
if len(timings):
    by_county = (
        timings.groupby(['stage', 'admin_id'])['seconds']
        .sum()
        .sort_values(ascending=False)
    )
    display(by_county.head(15))  # slowest (stage, county) pairs

# Snakemake job durations

Parsed from the newest run log.

- Start and finish per rule.
- Rules started but unfinished are the in-flight set.


In [ ]:
logs = sorted(SNAKEMAKE_LOG_DIR.glob('*.snakemake.log'))
jobs = pd.DataFrame()
if logs:
    text = logs[-1].read_text(encoding='utf-8', errors='ignore')
    stamp = r'\[\w{3} (\w{3} +\d+ [\d:]+) (\d{4})\]'
    starts = {}
    for m in re.finditer(stamp + r'\s*\n(?:local)?rule ([\w.-]+):', text):
        starts[m.group(3)] = datetime.strptime(
            f'{m.group(1)} {m.group(2)}', '%b %d %H:%M:%S %Y'
        )
    ends = {}
    for m in re.finditer(stamp + r'\s*\nFinished jobid: \d+ \(Rule: ([\w.-]+)\)', text):
        ends[m.group(3)] = datetime.strptime(
            f'{m.group(1)} {m.group(2)}', '%b %d %H:%M:%S %Y'
        )
    jobs = pd.DataFrame(
        [
            {
                'rule': k,
                'start': v,
                'end': ends.get(k),
                'minutes': ((ends[k] - v).total_seconds() / 60 if k in ends else None),
            }
            for k, v in starts.items()
        ]
    )
    in_flight = jobs[jobs['end'].isna()]
    print(f'{logs[-1].name}: {len(jobs)} rules started, {len(in_flight)} in flight')
    display(in_flight[['rule', 'start']].tail(10))
else:
    print('no snakemake logs found (driver-based run?)')

In [ ]:
if len(jobs):
    done = jobs.dropna(subset=['minutes'])
    display(done.sort_values('minutes', ascending=False)[['rule', 'minutes']].head(15))

# Driver log (phased builds)

The phased driver prints one line per job.

- Point `DRIVER_LOG` at its log for per-recipe rates.


In [ ]:
DRIVER_LOG = None  # e.g. Path('.../fast_nc.log')
if DRIVER_LOG and Path(DRIVER_LOG).exists():
    pat = re.compile(r'ok (\S+) (\S+) (\d+)s')
    rows = [
        {'recipe': m[0], 'admin_id': m[1], 'seconds': int(m[2])}
        for m in pat.findall(Path(DRIVER_LOG).read_text(errors='ignore'))
    ]
    df = pd.DataFrame(rows)
    display(
        df.groupby('recipe')['seconds']
        .agg(['count', 'median', 'sum'])
        .sort_values('sum', ascending=False)
    )

# Snakemake benchmarks x stage timers

The two persistent profiles, joined.

- Benchmarks: one tsv per rule with wall, CPU and max RSS.
- Timers: per-step JSON from inside the same job.
- The tsv says which jobs are slow; the JSON says which step.


In [ ]:
bench_dir = TIMER_LOG_ROOT / 'benchmarks'
rows = []
for f in sorted(bench_dir.glob('*.tsv')):
    try:
        b = pd.read_csv(f, sep='	')
    except Exception:
        continue
    stage, _, rest = f.stem.partition('_')
    recipe, _, admin = rest.rpartition('_')
    rows.append(
        {
            'stage': stage,
            'recipe': recipe,
            'admin_id': admin,
            'minutes': b['s'].iloc[0] / 60,
            'max_rss_mb': b.get('max_rss', pd.Series([None])).iloc[0],
        }
    )
bench = pd.DataFrame(rows)
if len(bench):
    display(bench.sort_values('minutes', ascending=False).head(12))
    if len(timings):
        slow = bench.iloc[0]
        steps = timings[timings['admin_id'] == slow['admin_id']]
        display(steps.sort_values('seconds', ascending=False).head(8))
else:
    print('no benchmark tsvs yet (written by Snakemake runs)')